# AI Cycling Coach — Local Fine-tune (M1 MPS)

Continues training from the main checkpoint, saves to a **separate** file (`cycling_coach_ft_local.pt`).
Does **not** touch `cycling_coach.pt`.

**Typical runtime on M1 Pro/Max:** ~1–2 hours for 10 extra epochs.

In [ ]:
# ── 1. Paths & device check ──────────────────────────────────────────────────
import os, sys, torch

REPO_ROOT = os.path.abspath('.')          # run from ai-coach/ root
for p in [os.path.join(REPO_ROOT, 'backend'), REPO_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ['PYTHONPATH'] = os.path.join(REPO_ROOT, 'backend')

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')
print(f'Repo    : {REPO_ROOT}')

In [ ]:
# ── 2. Generate small training dataset (skip if already fresh) ───────────────
import subprocess, multiprocessing, pandas as pd

DATA_FILE = 'ml/data/synthetic_ft.parquet'   # separate file so we don't clobber the big one
ATHLETES  = 5_000                             # ~30 sec to generate

os.makedirs('ml/data', exist_ok=True)

needs_gen = True
if os.path.exists(DATA_FILE):
    try:
        _s = pd.read_parquet(DATA_FILE, columns=['athlete_id', 'pc_5s_wkg'])
        print(f'✓ Found existing data: {_s.athlete_id.nunique():,} athletes — skipping generation')
        del _s
        needs_gen = False
    except Exception:
        print('⚠  Stale parquet — regenerating…')
        os.remove(DATA_FILE)

if needs_gen:
    workers = max(1, multiprocessing.cpu_count() - 2)
    print(f'Generating {ATHLETES:,} athletes with {workers} workers…')
    env = os.environ.copy()
    env['PYTHONPATH'] = os.path.join(REPO_ROOT, 'backend')
    subprocess.run([
        sys.executable, '-m', 'ml.training.generate_synthetic',
        '--athletes', str(ATHLETES),
        '--workers',  str(workers),
        '--output',   DATA_FILE,
    ], env=env, check=True)

df = pd.read_parquet(DATA_FILE)
print(f'✓ {len(df):,} rows | {df.athlete_id.nunique():,} athletes | {len(df.columns)} cols')
del df

In [ ]:
# ── 3. Fine-tune ─────────────────────────────────────────────────────────────
import argparse

CHECKPOINT  = 'backend/models/cycling_coach.pt'        # ← load from here
MODEL_FILE  = 'backend/models/cycling_coach_ft_local.pt'  # ← save here (NEVER overwrites main)

# ── Settings ─────────────────────────────────────────────────────────────────
EPOCHS           = 10    # epochs to add on top of the loaded checkpoint
BATCH_SIZE       = 256   # safe for M1 unified memory
STEPS_PER_EPOCH  = 500   # ~4 min/epoch on M1 Pro  →  ~40 min total
LEARNING_RATE    = 1e-4  # lower than pre-training (3e-4) for stable fine-tune

# Verify checkpoint exists
if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT}\nRun git pull first.')

_meta = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
_m    = _meta.get('metrics', {})
print(f'Loaded checkpoint: epoch {_m.get("epoch","?")}, '
      f'val_loss={_m.get("val_loss","?"):.4f}, '
      f'wt_acc={_m.get("wt_acc","?"): .1f}%')
print(f'Will train {EPOCHS} more epochs → save to {MODEL_FILE}')
print('─' * 60)

# Flush any cached module so edits to train.py take effect
for _k in list(sys.modules.keys()):
    if 'training.train' in _k:
        del sys.modules[_k]

os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)
from ml.training.train import train as run_training

args = argparse.Namespace(
    data             = DATA_FILE,
    output           = MODEL_FILE,
    checkpoint       = CHECKPOINT,   # load weights from main model
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    steps_per_epoch  = STEPS_PER_EPOCH,
    lr               = LEARNING_RATE,
    seq_len          = 90,
    val_frac         = 0.1,
    seed             = 42,
    d_model          = 256,
    nhead            = 8,
    num_layers       = 8,
    d_ff             = 1024,
    dropout          = 0.15,
    fast             = False,
    patience         = 8,
    compile          = False,
    no_amp           = True,    # MPS/CPU don't support CUDA AMP
)

run_training(args)
print(f'\n✓ Fine-tune complete → {MODEL_FILE}')

In [ ]:
# ── 4. Sanity check ──────────────────────────────────────────────────────────
from app.ml.model import CyclingTransformer

ckpt = torch.load(MODEL_FILE, map_location='cpu', weights_only=False)
cfg  = ckpt.get('config', {})
m    = CyclingTransformer(
    d_model         = cfg.get('d_model', 256),
    nhead           = cfg.get('nhead', 8),
    num_layers      = cfg.get('num_layers', 8),
    dim_feedforward = cfg.get('dim_feedforward', 1024),
)
m.load_state_dict(ckpt['state_dict'])
m.eval()

mt = ckpt.get('metrics', {})
print(f'✓ Model loaded: {sum(p.numel() for p in m.parameters()):,} params')
print(f'  epoch    : {mt.get("epoch",  "n/a")}')
print(f'  val_loss : {mt.get("val_loss","n/a")}')
print(f'  wt_acc   : {mt.get("wt_acc", "n/a")} %')
print(f'  IF_MAE   : {mt.get("if_mae", "n/a")}')